In [ ]:
import json

ALLOWED_ENGINES = {"qdrant", "weaviate", "milvus"}
target_engine = "qdrant"
ALLOWED_DATASETS = {
    "gist-960-euclidean",
    "dbpedia-openai-1M-1536-angular",
}

with open("cloud_exp.json", "r") as f:
    data = json.load(f)

filtered = [
    e for e in data
    if e.get("engine_name") in ALLOWED_ENGINES
    and e.get("dataset_name") in ALLOWED_DATASETS
    and e.get("hnsw_ef", 0.0) == 16
    and e.get("parallel", 0.0) == 100
]

with open("filtered.json", "w") as f:
    json.dump(filtered, f, indent=2)


In [ ]:
import json
from math import inf

ALLOWED_ENGINES = {"qdrant", "weaviate", "milvus"}
ALLOWED_DATASETS = {"gist-960-euclidean", "dbpedia-openai-1M-1536-angular"}

# You listed 5 target recalls, but said "4 entries per vector database".
# By default this will return ONE entry per target (so 5 per engine).
TARGET_RECALLS = [0.90, 0.925, 0.95, 0.9725, 1.0]
TARGET_RECALLS = [0.90, 0.95, 1.0]




def to_float(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


def iter_entries(data):
    """Yield entries from either a list[dict] JSON or dict[str, dict] JSON."""
    if isinstance(data, list):
        for e in data:
            if isinstance(e, dict):
                yield e
    elif isinstance(data, dict):
        for v in data.values():
            if isinstance(v, dict):
                yield v
    else:
        raise TypeError("Expected JSON top-level to be a list or dict.")


def filter_base(e):
    """Apply your earlier constraints."""
    if e.get("engine_name") not in ALLOWED_ENGINES:
        return False
    if e.get("parallel")  != 100:
        return False
    if e.get("dataset_name") not in ALLOWED_DATASETS:
        return False
    mp = to_float(e.get("mean_precisions"))
    if mp is None or mp <= 0.90:
        return False
   
    return True


def pick_closest_unique(entries, targets):
    """
    For each target, pick the entry whose mean_recall is closest.
    Ensures unique picks per target by not reusing the same object twice.
    """
    picks = []
    used_ids = set()

    # print(entries)
    for t in targets:
        best = None
        best_err = inf

        for e in entries:
           
            
            # Use object identity as a fallback uniqueness key
            uid = e.get("run_id") or e.get("id") or id(e)
            if uid in used_ids:
                continue

            mr = float(e["mean_recall"])  # safe because we filtered
            err = abs(mr - t)

            if err < best_err:
                best_err = err
                best = (uid, e, mr, err)

        if best is not None:
            uid, e, mr, err = best
            used_ids.add(uid)
            # Add helpful annotations (does not remove original fields)
            out = dict(e)
            out["_target_recall"] = t
            out["_recall_error"] = err
            picks.append(out)

    # sort by target recall for readability
    picks.sort(key=lambda x: x["_target_recall"])
    return picks


def main(in_path="input.json", out_path="picked.json"):
    with open(in_path, "r") as f:
        data = json.load(f)

    # Group by (engine, dataset)
    grouped = {(eng, ds): [] for eng in ALLOWED_ENGINES for ds in ALLOWED_DATASETS}

    for e in iter_entries(data):
        if not filter_base(e):
            continue
        e = dict(e)
        # normalize numerics once
        e["mean_recall"] = float(e["mean_precisions"])
        grouped[(e["engine_name"], e["dataset_name"])].append(e)

    # Pick 5 per dataset per engine
    result = {
        eng: {ds: pick_closest_unique(grouped[(eng, ds)], TARGET_RECALLS) for ds in ALLOWED_DATASETS}
        for eng in ALLOWED_ENGINES
    }

    with open(out_path, "w") as f:
        json.dump(result, f, indent=2)

    # quick console summary
    for eng in ALLOWED_ENGINES:
        for ds in ALLOWED_DATASETS:
            picks = result[eng][ds]
            print(f"{eng} | {ds}: {len(picks)} picks")
            for p in picks:
                
                print(
                    f"  target={p['_target_recall']:.4f} "
                    f"mean_recall={p['mean_recall']:.6f} "
                    f"upload={p['upload_time']:.6f} "
                    f"index={p['total_upload_time'] - p['upload_time']:.6f} "
                    f"RPS={p['rps']:.6f} "
                    f"Latency={1000 * p['mean_time']:.6f} "
                  
                )


if __name__ == "__main__":
    main(in_path="cloud_exp.json")


weaviate | dbpedia-openai-1M-1536-angular: 3 picks
  target=0.9000 mean_recall=0.943140 upload=836.963327 index=0.000056 RPS=1135.927164 Latency=3.843169 
  target=0.9500 mean_recall=0.945320 upload=843.320660 index=0.000027 RPS=360.556141 Latency=271.802702 
  target=1.0000 mean_recall=0.999360 upload=5162.220006 index=0.000018 RPS=171.580572 Latency=576.712750 
weaviate | gist-960-euclidean: 3 picks
  target=0.9000 mean_recall=0.907350 upload=4297.052306 index=0.000012 RPS=265.311314 Latency=359.683233 
  target=0.9500 mean_recall=0.947640 upload=881.557393 index=0.000014 RPS=202.513308 Latency=468.877737 
  target=1.0000 mean_recall=0.988260 upload=4297.052306 index=0.000012 RPS=157.304127 Latency=613.156499 
qdrant | dbpedia-openai-1M-1536-angular: 3 picks
  target=0.9000 mean_recall=0.900380 upload=235.411156 index=430.167090 RPS=1196.124133 Latency=9.706613 
  target=0.9500 mean_recall=0.950180 upload=224.446830 index=878.642126 RPS=1224.844133 Latency=3.701827 
  target=1.0000 m

In [42]:
import json

ALLOWED_ENGINES = {"qdrant", "weaviate", "milvus"}
ALLOWED_DATASETS = {
    "gist-960-euclidean",
    "dbpedia-openai-1M-1536-angular",
}
target_engine = "qdrant"
target_dataset = "gist-960-euclidean"

# 64, 256, 512
target_ef = 64
target_m = "m-64"

with open("cloud_exp.json", "r") as f:
    data = json.load(f)


filtered = []
for e in data:
    if e.get("engine_name") == target_engine:
        if e.get("dataset_name") == target_dataset:
            if e.get("parallel") == 100:
                if e.get("engine_params").get("hnsw_ef") == target_ef:
                    if e.get("engine_params").get("quantization",{}) == {}:
                        if target_m in e.get("setup_name"):
                            filtered.append(e)
                    


# filtered = [
#     e for e in data
#     if e.get("engine_name") == target_engine
#     and e.get("dataset_name") in ALLOWED_DATASETS
#     and e.get("engine_params",{}).get("hnsw_ef", 0.0) == 16
#     and e.get("parallel", 0.0) == 100
# ]

with open("filtered.json", "w") as f:
    json.dump(filtered, f, indent=2)
